<a href="https://colab.research.google.com/github/SameehaSyed05/flyrank-ml-internshipp/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SameehaSyed05/flyrank-ml-internshipp/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of analysis: Each row corresponds to a pseudonymized client, content item, and query hash combination in the search-performance dataset.

Time window: The table uses a fixed 90-day search-performance window. Where available, this window is split into last-30-day and previous-30-day measures. These sub-windows must be aligned before using any field for prediction, because information from an outcome period must not leak into the features.

In [11]:
!pip -q install duckdb

import duckdb
from google.colab import userdata

con = duckdb.connect()

hf_token = userdata.get("HF_TOKEN")

con.execute(f"""
CREATE SECRET hf_secret (
    TYPE huggingface,
    TOKEN '{hf_token}'
);
""")

rel = "hf://datasets/FlyRank/internship-warehouse"

query_table = f"{rel}/fact_content_query_90d.parquet"

# Total number of rows
print("Total rows:")
print(
    con.sql(f"""
        SELECT COUNT(*) AS total_rows
        FROM read_parquet('{query_table}')
    """).df()
)

# Check columns and data types
print("\nColumns and data types:")
print(
    con.sql(f"""
        DESCRIBE
        SELECT *
        FROM read_parquet('{query_table}')
    """).df()
)

Total rows:
   total_rows
0     2414248

Columns and data types:
                      column_name column_type null   key default extra
0                  client_hash_id     VARCHAR  YES  None    None  None
1                 content_hash_id     VARCHAR  YES  None    None  None
2                   query_hash_id     VARCHAR  YES  None    None  None
3                query_char_count      BIGINT  YES  None    None  None
4               query_token_count      BIGINT  YES  None    None  None
5                    window_start        DATE  YES  None    None  None
6                      window_end        DATE  YES  None    None  None
7                 impressions_90d      BIGINT  YES  None    None  None
8                      clicks_90d      BIGINT  YES  None    None  None
9              impressions_last30      BIGINT  YES  None    None  None
10                  clicks_last30      BIGINT  YES  None    None  None
11             impressions_prev30      BIGINT  YES  None    None  None
12          

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: Search-performance measurements from the historical or previous window, especially *_prev30 fields, are candidate features because they describe information available before the prediction point. Query-level search signals and safe historical aggregates may also be features after checking their measurement window.

Label / proxy: Any measure from the future or outcome window, including the target being predicted and fields directly calculated from that target, is a label or proxy and will not be used as a feature.

Context: client_hash_id, content_hash_id, and query_hash are context fields. They are used for grouping, joining, and splitting, but not as model features.

Excluded: Fields from the last-30-day outcome window are excluded when that same period defines the prediction outcome, because using them would leak future information. Repeated per-content context fields are also not summed because they repeat across query rows.

In [12]:
print(
    con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT client_hash_id) AS unique_clients,
        COUNT(DISTINCT content_hash_id) AS unique_content,
        COUNT(DISTINCT query_hash_id) AS unique_queries
    FROM read_parquet('{query_table}')
    """).df()
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  unique_clients  unique_content  unique_queries
0     2414248              52          133852         1180090


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Verification: I checked the table size, the uniqueness of the documented grain, the available columns, missing values, and the available measurement windows. Missingness is examined by category where appropriate because missing values may follow a systematic pattern rather than being random.

In [13]:
# Check the documented grain for duplicates
print(
    con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        query_hash_id,
        COUNT(*) AS n
    FROM read_parquet('{query_table}')
    GROUP BY
        client_hash_id,
        content_hash_id,
        query_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
    """).df()
)

# Preview a few rows
print(
    con.sql(f"""
    SELECT *
    FROM read_parquet('{query_table}')
    LIMIT 5
    """).df()
)

Empty DataFrame
Columns: [client_hash_id, content_hash_id, query_hash_id, n]
Index: []
            client_hash_id           content_hash_id           query_hash_id  \
0  client_08a6a72ff48e62c0  content_447894f2faf0d2bc  query_58b1b001f839d699   
1  client_08a6a72ff48e62c0  content_447894f2faf0d2bc  query_922b8eca2a24cd34   
2  client_08a6a72ff48e62c0  content_447894f2faf0d2bc  query_9f0c36a6ae2a6a99   
3  client_08a6a72ff48e62c0  content_447894f2faf0d2bc  query_a032820b5467e996   
4  client_08a6a72ff48e62c0  content_447894f2faf0d2bc  query_ba1a2f131961c5da   

   query_char_count  query_token_count window_start window_end  \
0                17                  3   2026-04-02 2026-06-30   
1                34                  7   2026-04-02 2026-06-30   
2                16                  2   2026-04-02 2026-06-30   
3                24                  4   2026-04-02 2026-06-30   
4                18                  3   2026-04-02 2026-06-30   

   impressions_90d  clicks_90d  imp

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Data limits: The dataset covers a fixed 90-day search-performance window from 2026-04-02 to 2026-06-30. Impression and click measures have no missing values, but average position is missing for 53,075 rows in the last-30-day window and 39,242 rows in the previous-30-day window. These missing values should be considered when comparing position-based performance. The fixed time window also means that the data cannot describe performance outside this period or establish longer-term trends.

In [14]:
print("Missing values:")
print(
    con.sql(f"""
    SELECT
        COUNT(*) - COUNT(impressions_90d) AS missing_impressions_90d,
        COUNT(*) - COUNT(clicks_90d) AS missing_clicks_90d,
        COUNT(*) - COUNT(impressions_last30) AS missing_impressions_last30,
        COUNT(*) - COUNT(clicks_last30) AS missing_clicks_last30,
        COUNT(*) - COUNT(impressions_prev30) AS missing_impressions_prev30,
        COUNT(*) - COUNT(clicks_prev30) AS missing_clicks_prev30,
        COUNT(*) - COUNT(avg_position_90d) AS missing_avg_position_90d,
        COUNT(*) - COUNT(avg_position_last30) AS missing_avg_position_last30,
        COUNT(*) - COUNT(avg_position_prev30) AS missing_avg_position_prev30
    FROM read_parquet('{query_table}')
    """).df()
)

print("\nTime windows:")
print(
    con.sql(f"""
    SELECT
        MIN(window_start) AS earliest_window_start,
        MAX(window_start) AS latest_window_start,
        MIN(window_end) AS earliest_window_end,
        MAX(window_end) AS latest_window_end
    FROM read_parquet('{query_table}')
    """).df()
)

Missing values:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   missing_impressions_90d  missing_clicks_90d  missing_impressions_last30  \
0                        0                   0                           0   

   missing_clicks_last30  missing_impressions_prev30  missing_clicks_prev30  \
0                      0                           0                      0   

   missing_avg_position_90d  missing_avg_position_last30  \
0                         0                       530758   

   missing_avg_position_prev30  
0                       329420  

Time windows:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  earliest_window_start latest_window_start earliest_window_end  \
0            2026-04-02          2026-04-02          2026-06-30   

  latest_window_end  
0        2026-06-30  


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.